# BirdCLEF 2026: Visual Spectrogram EDA

This notebook explores the conversion of raw audio files into Mel-spectrograms using our `SpectrogramGenerator`. We will visualize calls from different species and verify the visual quality before training our EfficientNet models.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import cv2
from src.audio.spectrograms import SpectrogramGenerator
import IPython.display as ipd

# Settings
DATA_DIR = "../data/raw"
TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, "train_audio")
SAMPLE_RATE = 32000
WINDOW_SIZE = 5

## 1. Initialize Generator

We use the parameters derived from the "Chirp Imagery" approach: 32kHz, 2048 FFT, and 512 hop length.

In [ ]:
spec_gen = SpectrogramGenerator(img_size=224)
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

## 2. Visualize Random Species

Let's look at a few different species to see how their "visual signatures" differ.

In [ ]:
def show_spectrogram_grid(df, n_samples=6):
    samples = df.sample(n_samples)
    
    plt.figure(figsize=(18, 10))
    for i, (_, row) in enumerate(samples.iterrows()):
        path = os.path.join(TRAIN_AUDIO_DIR, row['filename'])
        img = spec_gen.generate(path)
        
        plt.subplot(2, 3, i + 1)
        plt.imshow(img)
        plt.title(f"{row['common_name']}\n({row['primary_label']})")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

show_spectrogram_grid(train_df)

## 3. Comparing Calls within a Species

Do individuals of the same species have consistent visual patterns?

In [ ]:
target_species = train_df['primary_label'].value_counts().index[0]
species_df = train_df[train_df['primary_label'] == target_species]

print(f"Visualizing consistency for: {target_species}")
show_spectrogram_grid(species_df)

## 4. Audio Quality and Noise

The dataset contains a `rating` column. Let's see if there's a visual difference between high and low rating recordings.

In [ ]:
high_quality = train_df[train_df['rating'] >= 4.5].sample(3)
low_quality = train_df[train_df['rating'] <= 1.0].sample(3)
comparison = pd.concat([high_quality, low_quality])

plt.figure(figsize=(18, 10))
for i, (_, row) in enumerate(comparison.iterrows()):
    path = os.path.join(TRAIN_AUDIO_DIR, row['filename'])
    img = spec_gen.generate(path)
    
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    label_type = "High Quality" if i < 3 else "Low Quality"
    plt.title(f"{label_type} (Rating: {row['rating']})\n{row['primary_label']}")
    plt.axis('off')

plt.tight_layout()
plt.show()

## 5. Listening Test

Let's play one sample to hear what we are seeing.

In [ ]:
test_row = train_df.sample(1).iloc[0]
test_path = os.path.join(TRAIN_AUDIO_DIR, test_row['filename'])
audio, _ = librosa.load(test_path, sr=SAMPLE_RATE, duration=WINDOW_SIZE)
img = spec_gen.generate(test_path)

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title(f"Visual: {test_row['common_name']}")
plt.axis('off')
plt.show()

print(f"Audio: {test_row['common_name']}")
ipd.display(ipd.Audio(audio, rate=SAMPLE_RATE))